# Open Quantum Transport on Networks: From the Smallest Graph to the Current Lab

[Open in Colab](https://colab.research.google.com/github/pemodest0/Quantum-systems/blob/main/notebooks/colab_transport_tutorial_from_graph_to_lab.ipynb)

This notebook is the **teaching and presentation notebook**.

It starts from the minimal graph picture, explains what the excitation, sink, loss, disorder, and dephasing mean, and then runs small clean simulations with **only three helper files**:

- `notebooks/colab_transport_review/transport_core.py`
- `notebooks/colab_transport_review/transport_cases.py`
- `notebooks/colab_transport_review/transport_repo.py`

What this notebook is:

- a **classical numerical simulation** of an effective open quantum system;
- a clean path from simple examples to the current research questions;
- a presentation notebook that can be shown to a professor.

What this notebook is not:

- not a quantum-computer run;
- not a microscopic material simulation;
- not the full heavy atlas campaign.


## How to use this notebook

1. Run the setup cell.
2. Run the notebook from top to bottom.
3. Use the small examples first.
4. Only after that jump to the cells that load the real lab outputs.

The examples here are intentionally small enough to run quickly in Colab.


In [ ]:
%pip -q install numpy scipy pandas matplotlib networkx

import inspect
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("/content/Quantum-systems")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/pemodest0/Quantum-systems.git", str(ROOT)], check=True)

os.chdir(ROOT)
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from colab_transport_review import (
    choose_target,
    current_research_snapshot,
    dephasing_scan,
    draw_graph,
    make_graph,
    mini_seeded_campaign,
    quantum_vs_classical_case,
    reference_rows,
    simulate_open_quantum_transport,
    target_placement_scan,
)
from colab_transport_review import transport_cases, transport_core, transport_repo

plt.rcParams.update(
    {
        "figure.figsize": (8, 4.8),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
np.set_printoptions(precision=4, suppress=True)

print(f"Repository root: {ROOT}")


## 1. What the model means physically

We use an **effective one-excitation open-system model**.

- A **node** is a local quantum site.
- An **edge** is coherent hopping between two sites.
- The **excitation** is one packet of quantum amplitude placed on one node at the start.
- The **sink** is an absorbing target channel attached to one chosen node.
- **Loss** removes excitation from the graph without sending it to the target.
- **Disorder** means random onsite energy offsets.
- **Dephasing** means phase scrambling by the environment.

Central diagnostic:

\
S(\rho_g) = -\mathrm{Tr}(\rho_g \log \rho_g)
\

Here `rho_g` is the **graph-normalized remaining state on the graph**. This von Neumann entropy measures **mixing on the graph**, not transport success by itself.


In [ ]:
display(Markdown("### The three code files used in this notebook"))
for module in [transport_core, transport_cases, transport_repo]:
    path = Path(module.__file__)
    text = path.read_text(encoding="utf-8")
    print(f"\n{'=' * 90}\n{path.relative_to(ROOT)}\n{'=' * 90}")
    print(f"{len(text.splitlines())} lines")


## 2. Graph gallery


In [ ]:
gallery_families = [
    "chain", "ring", "star", "complete",
    "square_lattice_2d", "bottleneck", "clustered", "modular_two_community",
    "random_geometric", "erdos_renyi", "watts_strogatz_small_world", "barabasi_albert_scale_free",
    "sierpinski_gasket", "sierpinski_carpet_like",
]

ncols = 4
nrows = int(np.ceil(len(gallery_families) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows))
axes = axes.flatten()

for ax, family in zip(axes, gallery_families):
    graph, pos = make_graph(family, n=10, seed=11)
    initial_site = min(graph.nodes)
    target_site = max(graph.nodes)
    draw_graph(graph, pos, initial_site=initial_site, target_site=target_site, title=family, ax=ax)

for ax in axes[len(gallery_families):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 3. First experiment: one excitation on a chain

This is the simplest useful example:

- the excitation starts on the left;
- the target is on the right;
- we add a moderate amount of disorder and dephasing;
- we monitor target arrival, graph population, loss, entropy, and purity.


In [ ]:
graph, pos = make_graph("chain", n=6, seed=3)
initial_site = 0
target_site = 5

quantum, node_pop = simulate_open_quantum_transport(
    graph,
    pos,
    initial_site=initial_site,
    target_site=target_site,
    disorder_strength_over_J=0.3,
    gamma_phi_over_J=0.1,
    seed=3,
    t_final=12,
    n_times=140,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
draw_graph(graph, pos, initial_site=initial_site, target_site=target_site, title="Chain: initial site 0, target site 5", ax=axes[0, 0])

axes[0, 1].plot(quantum["time"], node_pop)
axes[0, 1].set_title("Population on graph sites")
axes[0, 1].set_xlabel("time")
axes[0, 1].set_ylabel("population")

axes[1, 0].plot(quantum["time"], quantum["target_arrival"], label="target arrival", color="#d62728")
axes[1, 0].plot(quantum["time"], quantum["loss"], label="loss", color="#555555")
axes[1, 0].plot(quantum["time"], quantum["graph_population"], label="still in graph", color="#1f77b4")
axes[1, 0].set_title("Population closure")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("probability")
axes[1, 0].legend()

axes[1, 1].plot(quantum["time"], quantum["von_neumann_entropy"], label="von Neumann entropy", color="#2ca02c")
axes[1, 1].plot(quantum["time"], quantum["purity"], label="purity", color="#9467bd")
axes[1, 1].set_title("Mixing diagnostics")
axes[1, 1].set_xlabel("time")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

quantum.tail()


### How to read the chain plots

- **Target arrival** is the useful quantity: how much reached the target channel.
- **Loss** is useless transport: excitation that disappeared before reaching the target.
- **Graph population** is what still remains on the graph.
- **Von Neumann entropy** tells you how mixed the graph state is.
- **Purity** falls as the state becomes more mixed.

Entropy going up is **not automatically good**. It can happen together with good transport, bad transport, or simple spreading.


## 4. Phase scrambling scan


In [ ]:
scan = pd.concat(
    [dephasing_scan(family, n=8, W=0.6, seed=3)[0] for family in ["chain", "ring", "complete", "star"]],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for family, group in scan.groupby("family"):
    axes[0].plot(group["gamma_phi_over_J"], group["target_arrival"], marker="o", label=family)
    axes[1].plot(group["gamma_phi_over_J"], group["von_neumann_entropy"], marker="o", label=family)

axes[0].set_title("Target arrival versus phase scrambling")
axes[0].set_xlabel("gamma_phi / J")
axes[0].set_ylabel("target arrival")
axes[0].legend()

axes[1].set_title("Final graph-normalized von Neumann entropy")
axes[1].set_xlabel("gamma_phi / J")
axes[1].set_ylabel("entropy")
axes[1].legend()

plt.tight_layout()
plt.show()

scan.groupby("family")[["target_arrival", "gain_over_zero_dephasing", "von_neumann_entropy", "purity"]].max().round(4)


The physical question here is **not** “does entropy increase?”.

The correct question is:

**Does a nonzero dephasing rate increase useful arrival at the target compared with the zero-dephasing case?**


## 5. Same graph, different target


In [ ]:
placement, ring_graph, ring_pos, ring_initial = target_placement_scan(
    "ring",
    n=8,
    target_styles=("near", "far", "high_centrality", "low_centrality"),
    W=0.6,
    gamma=0.1,
    seed=3,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
draw_graph(
    ring_graph,
    ring_pos,
    initial_site=ring_initial,
    target_site=int(placement.loc[placement["target_style"] == "far", "target_site"].iloc[0]),
    title="Ring example with one highlighted target",
    ax=axes[0],
)
axes[1].bar(placement["target_style"], placement["target_arrival"], color="#4c78a8")
axes[1].set_title("Target arrival for different target placements")
axes[1].set_ylabel("target arrival")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

placement[["target_style", "target_site", "target_arrival", "von_neumann_entropy", "target_degree", "shortest_path_distance", "target_closeness"]].round(4)


This is one of the central ideas of the project:

**the same network can behave very differently when only the target position changes**.


## 6. Quantum versus classical control


In [ ]:
merged, summary, qc_graph, qc_pos, qc_initial, qc_target = quantum_vs_classical_case(
    "ring",
    n=8,
    W=0.6,
    gamma=0.1,
    seed=3,
    target_style="far",
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
draw_graph(qc_graph, qc_pos, initial_site=qc_initial, target_site=qc_target, title="Same graph, same initial site, same target", ax=axes[0])
axes[1].plot(merged["time"], merged["target_arrival"], label="open quantum model", color="#1f77b4")
axes[1].plot(merged["time"], merged["target_arrival_classical"], label="classical rate model", color="#ff7f0e")
axes[1].set_title("Target arrival: quantum versus classical")
axes[1].set_xlabel("time")
axes[1].set_ylabel("target arrival")
axes[1].legend()
plt.tight_layout()
plt.show()

pd.DataFrame([summary]).round(4)


A classical control prevents a false claim.

If the classical model reaches the target equally well, then the result is mostly a **network-connectivity effect**, not a specifically quantum one.


## 7. Small seeded campaign


In [ ]:
campaign = mini_seeded_campaign(
    families=("chain", "ring", "star", "complete"),
    n=8,
    seeds=(3, 5, 7, 11),
    gamma_values=(0.0, 0.05, 0.1, 0.2, 0.4),
    W=0.6,
)

summary = campaign.groupby("family", as_index=False).agg(
    arrival_mean=("target_arrival", "mean"),
    entropy_mean=("von_neumann_entropy", "mean"),
    purity_mean=("purity", "mean"),
    best_gain=("gain_over_zero_dephasing", "max"),
)
summary = summary.sort_values("arrival_mean", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
axes[0].bar(summary["family"], summary["arrival_mean"], color="#4c78a8")
axes[0].set_title("Mean target arrival")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(summary["family"], summary["entropy_mean"], color="#72b7b2")
axes[1].set_title("Mean final von Neumann entropy")
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(summary["family"], summary["best_gain"], color="#54a24b")
axes[2].axhline(0.05, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Best dephasing gain over zero-noise")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

summary.round(4)


## 8. Load the real lab state


In [ ]:
snapshot = current_research_snapshot(ROOT)
evidence_metrics = snapshot["evidence_prep_metrics"]
journey_metrics = snapshot["research_journey_metrics"]
classification_metrics = snapshot["classification_metrics"]

display(Markdown("### Evidence-prep atlas metrics"))
display(pd.DataFrame([evidence_metrics]))

display(Markdown("### Research journey metrics"))
display(pd.DataFrame([journey_metrics]))

display(Markdown("### Network classification metrics"))
display(pd.DataFrame([classification_metrics]))


## 9. What is already strong and what is still open

Already strong enough to discuss:

- target placement matters;
- some graph families show positive quantum-minus-classical arrival;
- moderate dephasing can improve useful arrival in selected regimes;
- entropy and purity work well as diagnostics of mixing.

Still open:

- the full strong atlas is not finished yet;
- the intense atlas is not the right run for a mixed-use machine;
- not every family boundary is resolved;
- entropy must never be sold as transport success by itself.


## 10. References


In [ ]:
refs = reference_rows()
refs
